# CNN Hyperparameter Tuning

This notebook runs five controlled CNN experiments using the same leakage-safe GTZAN train/validation/test split.

**Do not change the test split between experiments.**


In [ ]:
# 1. Clone the latest project branch
!rm -rf /content/music-genre-classification
!git clone -b feature/deep-learning https://github.com/SaimaKapoor/music-genre-classification.git /content/music-genre-classification


Cloning into '/content/music-genre-classification'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 46 (delta 11), reused 37 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 61.80 KiB | 1.93 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [ ]:
# 2. Imports and path setup
import sys
from pathlib import Path
import pandas as pd
import torch

REPO = Path('/content/music-genre-classification')
sys.path.insert(0, str(REPO))

from src.deep_learning.dataset import GTZANMelDataset
from src.deep_learning.cnn import BaselineCNN
from src.deep_learning.train import fit
from src.deep_learning.evaluate import predict, classification_metrics

print('IMPORTS SUCCESSFUL!')
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


IMPORTS SUCCESSFUL!
Device: cuda


In [ ]:
import os

RAW_URL = "https://ibm.ent.box.com/index.php?rm=box_download_shared_file&shared_name=gvkgmb4n0h6rbeccdwtwa0ujjjfmouof&file_id=f_961929491012"
ARCHIVE = "/content/genres.tar.gz"

if not os.path.exists("/content/gtzan/genres"):
    print("Downloading GTZAN...")
    !wget -O /content/genres.tar.gz "$RAW_URL"

    print("Extracting GTZAN...")
    !mkdir -p /content/gtzan
    !tar -xzf /content/genres.tar.gz -C /content/gtzan

print("GTZAN ready!")
!find /content/gtzan -maxdepth 2 -type d | head -20

GTZAN ready!
/content/gtzan
/content/gtzan/genres
/content/gtzan/genres/blues
/content/gtzan/genres/classical
/content/gtzan/genres/metal
/content/gtzan/genres/pop
/content/gtzan/genres/disco
/content/gtzan/genres/rock
/content/gtzan/genres/hiphop
/content/gtzan/genres/reggae
/content/gtzan/genres/country
/content/gtzan/genres/jazz


In [ ]:
from pathlib import Path

MANIFEST = Path("/content/split_manifest.csv")

print("GTZAN exists:", Path("/content/gtzan").exists())
print("Manifest exists:", MANIFEST.exists())

GTZAN exists: True
Manifest exists: False


In [ ]:
# 3. Locate the dataset and manifest
from google.colab import files

MANIFEST = Path('/content/split_manifest.csv')
if not MANIFEST.exists():
    print('split_manifest.csv not found. Upload it now.')
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    MANIFEST = Path('/content') / uploaded_name

# Baseline notebook uses /content/gtzan/genres. Accept either that or /content/gtzan.
possible_roots = [Path('/content/gtzan/genres'), Path('/content/gtzan')]
AUDIO_ROOT = next((p for p in possible_roots if p.exists()), None)

if AUDIO_ROOT is None:
    raise FileNotFoundError(
        'GTZAN audio was not found. Run the dataset-download/setup section from 04_CNN_Baseline.ipynb first, then rerun this notebook.'
    )

manifest = pd.read_csv(MANIFEST)
print('Manifest:', MANIFEST)
print('Audio root:', AUDIO_ROOT)
print('Rows:', len(manifest))
print(manifest['split'].value_counts())


split_manifest.csv not found. Upload it now.


Saving split_manifest.csv to split_manifest.csv
Manifest: /content/split_manifest.csv
Audio root: /content/gtzan/genres
Rows: 1000
split
train         700
test          150
validation    150
Name: count, dtype: int64


In [ ]:
# 4. Copy the tuning runner into the cloned repo
from pathlib import Path
from google.colab import files

# Upload tuning.py if it is not already available in this runtime.
TUNING_SRC = Path('/content/tuning.py')
if not TUNING_SRC.exists():
    print('Upload tuning.py from the corrected ZIP.')
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    TUNING_SRC = Path('/content') / uploaded_name

TUNING_DST = REPO / 'tuning.py'
TUNING_DST.write_text(TUNING_SRC.read_text(encoding='utf-8'), encoding='utf-8')
print('Tuning runner ready:', TUNING_DST)


Upload tuning.py from the corrected ZIP.


Saving tuning.py to tuning.py
Tuning runner ready: /content/music-genre-classification/tuning.py


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving tuning.py to tuning (1).py


In [ ]:
import shutil

shutil.copy(
    "/content/tuning.py",
    "/content/music-genre-classification/tuning.py"
)

print("V2 tuning runner installed!")

V2 tuning runner installed!


In [ ]:
import sys

sys.path.insert(0, "/content/music-genre-classification")

if "tuning" in sys.modules:
    del sys.modules["tuning"]

from tuning import run_all

print("V2 tuning runner loaded!")

V2 tuning runner loaded!


In [ ]:
# 5. Run all five controlled experiments
from tuning import run_all

results_df, histories = run_all(
    manifest_path=str(MANIFEST),
    audio_root=str(AUDIO_ROOT),
    epochs=15,
    seed=42,
)

print('\nFINAL TUNING RESULTS')
display(results_df)

results_df.to_csv('/content/cnn_tuning_results.csv', index=False)
print('\nSaved /content/cnn_tuning_results.csv')


Resolved audio root: /content/gtzan
Matched 50/50 manifest paths during root check.

Running: baseline
Songs: train=700, val=150, test=150
Device: cuda
Epoch 01/15 | train loss 2.2969 | train acc 0.1100 | val loss 2.2627 | val acc 0.1333
Epoch 02/15 | train loss 2.2013 | train acc 0.1729 | val loss 2.1046 | val acc 0.2200
Epoch 03/15 | train loss 2.0948 | train acc 0.2171 | val loss 1.9640 | val acc 0.2867
Epoch 04/15 | train loss 2.0143 | train acc 0.2629 | val loss 1.8591 | val acc 0.3133
Epoch 05/15 | train loss 1.9479 | train acc 0.2871 | val loss 1.7995 | val acc 0.3267
Epoch 06/15 | train loss 1.9047 | train acc 0.2943 | val loss 1.7601 | val acc 0.3533
Epoch 07/15 | train loss 1.8293 | train acc 0.3157 | val loss 1.6111 | val acc 0.4000
Epoch 08/15 | train loss 1.7448 | train acc 0.3657 | val loss 1.5502 | val acc 0.4200
Epoch 09/15 | train loss 1.6788 | train acc 0.3857 | val loss 1.4974 | val acc 0.4533
Epoch 10/15 | train loss 1.6665 | train acc 0.3971 | val loss 1.4817 | val

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


RESULT | high_dropout | best val acc=0.4867 | test acc=0.4200 | test macro-F1=0.3447

Running: larger_batch
Songs: train=700, val=150, test=150
Device: cuda
Epoch 01/15 | train loss 2.3043 | train acc 0.0971 | val loss 2.2848 | val acc 0.1667
Epoch 02/15 | train loss 2.2595 | train acc 0.1586 | val loss 2.1301 | val acc 0.2467
Epoch 03/15 | train loss 2.1283 | train acc 0.2343 | val loss 1.9899 | val acc 0.2667
Epoch 04/15 | train loss 2.0589 | train acc 0.2457 | val loss 1.9335 | val acc 0.3267
Epoch 05/15 | train loss 2.0223 | train acc 0.2514 | val loss 1.9057 | val acc 0.3067
Epoch 06/15 | train loss 2.0077 | train acc 0.2657 | val loss 1.8178 | val acc 0.3133
Epoch 07/15 | train loss 1.9403 | train acc 0.2871 | val loss 1.7915 | val acc 0.3400
Epoch 08/15 | train loss 1.8701 | train acc 0.3157 | val loss 1.6686 | val acc 0.3600
Epoch 09/15 | train loss 1.8055 | train acc 0.3371 | val loss 1.6552 | val acc 0.4133
Epoch 10/15 | train loss 1.7572 | train acc 0.3457 | val loss 1.5554 

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,experiment,learning_rate,dropout,batch_size,best_val_accuracy,test_accuracy,test_macro_f1
0,baseline,0.0010,0.3,32,0.520000,0.480000,0.450544
1,low_lr,0.0005,0.3,32,0.506667,0.520000,0.485316
2,high_dropout,0.0010,0.5,32,0.486667,0.420000,0.344682
3,larger_batch,0.0010,0.3,64,0.486667,0.486667,0.459377
4,low_lr_high_dropout,0.0005,0.5,32,0.413333,0.386667,0.312881



Saved /content/cnn_tuning_results.csv


In [ ]:
# 6. Save the best configuration
best = results_df.iloc[0]
print('BEST CONFIGURATION')
print(best.to_string())


BEST CONFIGURATION
experiment           baseline
learning_rate           0.001
dropout                   0.3
batch_size                 32
best_val_accuracy        0.52
test_accuracy            0.48
test_macro_f1        0.450544


## What to send back

Send the **FINAL TUNING RESULTS** table. We will select the best configuration based on validation/test performance and then train the final CNN properly before doing architecture and representation experiments.
